# 14. 점포 → SK 유동인구 격자 연결

## 왜 하나
가게(상가정보) 위치와 유동인구(SK, 50m 격자)를 같은 칸으로 묶어야
"어느 동네에 어떤 가게가 몰려 있고, 사람이 얼마나 다니는지"를 함께 볼 수 있다.

## 방법
- SK 격자 좌표(X_COORD, Y_COORD)는 50m 칸의 **가운데 점**이다 (EPSG:5179)
- 가게 좌표(x, y)에서 **가장 가까운 격자 가운데 점**을 찾는다
- 거리가 36m 이내면 그 칸 안의 가게로 본다 (50m 칸의 가운데에서 꼭짓점까지 약 35.4m)
- 격자 키는 (X_COORD, Y_COORD)를 쓴다. BLOCK_CD는 여러 칸을 묶은 소지역 코드라 칸을 구분하지 못한다
- 골프장·레고랜드(출처 = 수동추가)는 가게 수에 넣지 않고 격자별 "있음" 표시로 따로 쓴다

## 저장
- `stores_grid.csv` (가게별 격자)
- `grid_features.csv` (격자별 지표)
- `grid_rule_types.csv` (규칙 기반 유형, 검증용)
- `grid_types.csv` (K-평균 최종 유형)
- 그림: `14_grid_types_map.png`, `14_grid_type_profile.png`

## 1. 준비

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!apt-get -qq install -y fonts-nanum > /dev/null

import json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.spatial import cKDTree

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
OUT = BASE / "data" / "processed"
FIG = BASE / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

## 2. 격자 불러오기

In [ ]:
SKC = OUT / "sk_coordinate"
files = {"강남": "SK_Gangnam_Coordinates_202507_202512.csv",
         "춘천": "SK_Chuncheon_Coordinates_202507_202512.csv"}

grids = []
for region, fname in files.items():
    g = pd.read_csv(SKC / fname)
    g = g.drop_duplicates(["X_COORD", "Y_COORD"])[["X_COORD", "Y_COORD"]].copy()
    g["region"] = region
    grids.append(g)
grid = pd.concat(grids, ignore_index=True)

print("격자 수 (6개월 통합, 좌표 기준 중복 제거):")
print(grid.groupby("region").size())

## 3. 가게 → 격자 연결 + 저장

In [ ]:
stores = pd.read_csv(OUT / "stores_mapped_with_golf.csv", low_memory=False)
print("지역 값:", stores["region"].value_counts().to_dict())

res = []
for region in ["강남", "춘천"]:
    s = stores[stores["region"] == region].copy()
    g = grid[grid["region"] == region].reset_index(drop=True)
    tree = cKDTree(g[["X_COORD", "Y_COORD"]].values)
    dist, idx = tree.query(s[["x", "y"]].values)       # 가장 가까운 격자 가운데 점
    s["grid_x"] = g["X_COORD"].values[idx]
    s["grid_y"] = g["Y_COORD"].values[idx]
    s["grid_dist"] = dist.round(1)
    res.append(s)

sg = pd.concat(res, ignore_index=True)
sg["in_grid"] = sg["grid_dist"] <= 36
sg["grid_id"] = sg["region"] + "_" + sg["grid_x"].round(0).astype(int).astype(str) + "_" + sg["grid_y"].round(0).astype(int).astype(str)

sg.to_csv(OUT / "stores_grid.csv", index=False, encoding="utf-8-sig")
print("저장 완료: stores_grid.csv", f"({len(sg):,}행)")

## 4. 연결 확인
- 격자 안 비율이 거의 1.0이고, 가게–격자 중심 평균 거리가 약 19m면 정상 (격자 좌표 = 50m 칸의 가운데 점)

In [ ]:
shop = sg[sg["출처"] == "상가정보"]

print("① 격자 안에 들어간 가게 비율")
print(shop.groupby("region")["in_grid"].mean().round(3))

print("\n② 가게–격자 중심 거리(m)")
print(shop.groupby("region")["grid_dist"].describe(percentiles=[.5, .9, .99]).round(1))

print("\n③ 가게가 1곳 이상 있는 격자 수")
print(shop[shop["in_grid"]].groupby("region")["grid_id"].nunique())

print("\n④ 골프장·레고랜드")
print(sg[sg["출처"] != "상가정보"][["상호명", "grid_dist", "in_grid"]].to_string(index=False))

## 5. 격자 지표

| 지표 | 계산 (2025.7~12 월평균) | 의미 |
|---|---|---|
| pop | 성·연령별 유동인구 합 | 하루 평균 유동인구 |
| elder_share | 60대 이상 ÷ 전체 | 폭염 취약층 비중 |
| weekend_ratio | 토·일 평균 ÷ 월~금 평균 | 1보다 크면 주말에 사람이 몰림 |
| work_share | 9~17시 ÷ 하루 전체 | 낮 시간 집중 |
| night_share | 18~23시 ÷ 하루 전체 | 저녁 시간 집중 |
| food_share / tour_share | (음식점+카페) / (야외레저+숙박) ÷ 가게 수 | 설명용 (군집에는 미사용) |
| has_golf / has_lego | 골프장·레고랜드 있음 | 호우 취약 시설 |

In [ ]:
SKP = OUT / "sk_processed"
KEY = ["X_COORD", "Y_COORD"]
DROP = ["STD_YM", "BLOCK_CD", "X_COORD", "Y_COORD", "REGION"]

def load_flow(region_en, region_kr):
    out = None
    for kind in ["age", "wkdy", "time"]:
        d = pd.read_csv(SKP / region_en / f"flow_{kind}_{region_en}_202507_202512.csv")
        d[KEY] = d[KEY].round(2)
        num = [c for c in d.columns if c not in DROP]
        # 같은 달·같은 칸은 합치고(소지역 경계에 걸친 칸) → 6개월 평균
        d = d.groupby(["STD_YM"] + KEY)[num].sum().groupby(level=KEY).mean()
        out = d if out is None else out.join(d, how="outer")
    out = out.reset_index()
    out["region"] = region_kr
    return out

flow = pd.concat([load_flow("gangnam", "강남"), load_flow("chuncheon", "춘천")], ignore_index=True)

age = [c for c in flow.columns if c.startswith(("MAN_", "WMAN_"))]
wkd = [f"FLOW_POP_CNT_{d}" for d in ["MON", "TUS", "WED", "THU", "FRI"]]
wke = ["FLOW_POP_CNT_SAT", "FLOW_POP_CNT_SUN"]
tm  = [f"TMST_{h:02d}" for h in range(24)]

f = flow[["region"] + KEY].copy()
f["pop"] = flow[age].sum(axis=1)
f["elder_share"] = (flow["MAN_FLOW_POP_CNT_60GU"] + flow["WMAN_FLOW_POP_CNT_60GU"]) / f["pop"].replace(0, np.nan)
f["weekend_ratio"] = flow[wke].mean(axis=1) / flow[wkd].mean(axis=1).replace(0, np.nan)
t_all = flow[tm].sum(axis=1).replace(0, np.nan)
f["work_share"] = flow[[f"TMST_{h:02d}" for h in range(9, 18)]].sum(axis=1) / t_all
f["night_share"] = flow[[f"TMST_{h:02d}" for h in range(18, 24)]].sum(axis=1) / t_all

print("격자 수:", f.groupby("region").size().to_dict())
print(f.groupby("region")[["pop", "elder_share", "weekend_ratio", "work_share", "night_share"]].median().round(3))

## 6. 가게 구성 붙이기 + 저장 (`grid_features.csv`)

In [ ]:
sg = pd.read_csv(OUT / "stores_grid.csv", low_memory=False)
sg["X_COORD"] = sg["grid_x"].round(2)
sg["Y_COORD"] = sg["grid_y"].round(2)
G = ["region"] + KEY

shop = sg[(sg["출처"] == "상가정보") & sg["in_grid"]]
comp = shop.groupby(G).agg(
    n_stores=("상호명", "size"),
    n_food=("ind_grp", lambda s: s.isin(["음식점", "카페·제과"]).sum()),
    n_tour=("ind_grp", lambda s: s.isin(["야외레저", "숙박"]).sum()),
).reset_index()

extra = sg[sg["출처"].str.startswith("수동추가")]
golf = extra[extra["출처"] == "수동추가_골프장"].groupby(G).size().rename("has_golf").reset_index()
lego = extra[extra["출처"] == "수동추가_관광시설"].groupby(G).size().rename("has_lego").reset_index()

feat = (f.merge(comp, on=G, how="left")
         .merge(golf, on=G, how="left")
         .merge(lego, on=G, how="left"))
feat[["n_stores", "n_food", "n_tour", "has_golf", "has_lego"]] = feat[["n_stores", "n_food", "n_tour", "has_golf", "has_lego"]].fillna(0).astype(int)
feat["has_golf"] = (feat["has_golf"] > 0).astype(int)
feat["has_lego"] = (feat["has_lego"] > 0).astype(int)
feat["food_share"] = feat["n_food"] / feat["n_stores"].replace(0, np.nan)
feat["tour_share"] = feat["n_tour"] / feat["n_stores"].replace(0, np.nan)

feat.to_csv(OUT / "grid_features.csv", index=False, encoding="utf-8-sig")

# 확인: 가게가 SK 격자와 제대로 붙었는지
print("가게 붙은 격자:", (feat["n_stores"] > 0).groupby(feat["region"]).sum().to_dict(), "(stores_grid 결과: 강남 4,354 / 춘천 4,638)")
print("붙은 가게 수:", feat.groupby("region")["n_stores"].sum().to_dict())
print("골프장 격자:", feat["has_golf"].sum(), "| 레고랜드 격자:", feat["has_lego"].sum())

## 7. 규칙 기반 분류 (검증용)
주말/평일 비율 하나로 나눈 단순 기준. 뒤의 K-평균 결과가 상식과 맞는지 비교하는 데 쓴다.
- 관광형: 주말 비율 ≥ 1.2 또는 골프장·레고랜드
- 업무형: 주말 비율 ≤ 0.7
- 생활형: 나머지

In [ ]:
TOUR_TH, WORK_TH = 1.2, 0.7

base = feat[((feat["n_stores"] >= 3) | (feat["has_golf"] == 1) | (feat["has_lego"] == 1)) & (feat["pop"] > 0)].copy()
base["log_pop"] = np.log1p(base["pop"])

def classify(r, tour_th=TOUR_TH, work_th=WORK_TH):
    if r["has_golf"] == 1 or r["has_lego"] == 1 or r["weekend_ratio"] >= tour_th:
        return "관광형"
    if r["weekend_ratio"] <= work_th:
        return "업무형"
    return "생활형"

base["rule_type"] = base.apply(classify, axis=1)
print(pd.crosstab(base["rule_type"], base["region"]))

print("\n[민감도] 관광형 격자 수")
for th in [1.1, 1.2, 1.3]:
    print(f"  주말 비율 ≥ {th}:", base.apply(lambda r: classify(r, tour_th=th), axis=1).eq("관광형").groupby(base["region"]).sum().to_dict())
print("[민감도] 업무형 격자 수")
for th in [0.6, 0.7, 0.8]:
    print(f"  주말 비율 ≤ {th}:", base.apply(lambda r: classify(r, work_th=th), axis=1).eq("업무형").groupby(base["region"]).sum().to_dict())

base[G + ["rule_type"]].to_csv(OUT / "grid_rule_types.csv", index=False, encoding="utf-8-sig")

## 8. K-평균 군집
- 대상: 가게 3곳 이상 또는 골프장·레고랜드가 있는 격자
- 변수: 유동인구 지표만 (log 유동인구, 주말/평일, 낮 시간, 저녁, 60대 이상). 가게 비율은 가게 수가 적은 칸에서 값이 듬성듬성해 제외
- 지역별 표준화, 상·하위 1% 절단, 군집 수는 3~6 중 실루엣 최대

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

FLOW = ["log_pop", "weekend_ratio", "work_share", "night_share", "elder_share"]   # 유동인구 변수만


res = []
for region in ["강남", "춘천"]:
    d = base[base["region"] == region].copy()
    X = d[FLOW].fillna(d[FLOW].median())
    X = X.clip(X.quantile(0.01), X.quantile(0.99), axis=1)
    Z = StandardScaler().fit_transform(X)

    # 군집 수 3~6 중 실루엣 비교
    scores = {k: silhouette_score(Z, KMeans(k, n_init=20, random_state=42).fit_predict(Z)) for k in range(3, 7)}
    best_k = max(scores, key=scores.get)
    print(f"\n[{region}] 대상 {len(d):,}개 | 실루엣: " + ", ".join(f"k={k} {v:.3f}" for k, v in scores.items()) + f" → k={best_k}")

    d["cluster"] = KMeans(best_k, n_init=20, random_state=42).fit_predict(Z)

    # 군집별 평균표 (이걸 보고 이름을 붙인다)
    prof = d.groupby("cluster").agg(격자수=("pop", "size"), 가게수=("n_stores", "sum"),
                                     유동인구=("pop", "median"), 주말비율=("weekend_ratio", "median"),
                                     업무시간=("work_share", "median"), 저녁=("night_share", "median"),
                                     고령비중=("elder_share", "median"), 외식비중=("food_share", "median"),
                                     관광비중=("tour_share", "median"), 골프장=("has_golf", "sum"), 레고=("has_lego", "sum"))
    print(prof.round(3).to_string())
    res.append(d)

km_df = pd.concat(res, ignore_index=True)
km_df.to_csv(OUT / "grid_kmeans_raw.csv", index=False, encoding="utf-8-sig")

## 9. 이름 붙이기 + 규칙 기반과 비교 + 저장 (`grid_types.csv`)
이름은 위 군집별 평균표를 보고 정했다.

In [ ]:
NAMES = {
    "강남": {0: "평일·업무형", 1: "주거·고령형", 2: "번화가형", 3: "주말·여가형"},
    "춘천": {0: "주말·여가형", 1: "낮·고령형", 2: "시내 생활형"},
}
km_df["type"] = km_df.apply(lambda r: NAMES[r["region"]][r["cluster"]], axis=1)

rule = pd.read_csv(OUT / "grid_rule_types.csv")
for c in KEY:
    rule[c] = rule[c].round(2)
cmp = km_df.drop(columns="rule_type", errors="ignore").merge(rule, on=G, how="left")
for region in ["강남", "춘천"]:
    c = cmp[cmp["region"] == region]
    print(f"\n[{region}] K-평균(행) vs 규칙 기반(열)")
    print(pd.crosstab(c["type"], c["rule_type"]))

out = km_df[G + ["cluster", "type", "pop", "weekend_ratio", "work_share", "night_share",
                 "elder_share", "n_stores", "food_share", "tour_share", "has_golf", "has_lego"]].copy()
out.to_csv(OUT / "grid_types.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료: grid_types.csv")

## 10. 행정동 경계 받기
지도 바탕용. vuski/admdongkor (행정동 경계 2025-01-01, 통계청 자료 기반) → 강남구·춘천시만 `data/external/adm/`에 저장 (처음 한 번만)

In [ ]:
ADM_DIR = BASE / "data/external/adm"
ADM_DIR.mkdir(parents=True, exist_ok=True)
ADM_PATH = ADM_DIR / "adm_gangnam_chuncheon_2025.geojson"

if not ADM_PATH.exists():
    url = "https://raw.githubusercontent.com/vuski/admdongkor/master/ver20250101/HangJeongDong_ver20250101.geojson"
    allgj = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    feats = [f for f in allgj["features"] if ("춘천시" in f["properties"]["adm_nm"] or "강남구" in f["properties"]["adm_nm"])]
    json.dump({"type": "FeatureCollection", "features": feats}, open(ADM_PATH, "w", encoding="utf-8"), ensure_ascii=False)
print("행정동 경계:", ADM_PATH)

## 11. 유형 지도
유형마다 한 장씩: 그 유형 칸만 색, 나머지는 회색. 바탕은 행정동 경계.

In [ ]:
GT_PATH = OUT / "grid_types.csv"
SAVE = FIG / "14_grid_types_map.png"

from matplotlib.patches import Polygon as MPoly
from matplotlib.collections import PatchCollection
from pyproj import Transformer

INK, SUB, LAND, DOT = "#1d2430", "#5b6472", "#eceef1", "#c3c9d2"
TYPES = {   # 지역: [(유형, 색, 한 줄 설명, (표시할 지명, 위도, 경도))]
    "강남": [("평일·업무형", "#1f5fa8", "평일 낮 직장인 중심, 주말엔 한산", ("테헤란로", 37.5006, 127.0364)),
             ("번화가형",   "#e07b39", "유동인구·저녁 인구 최다",        ("강남역", 37.4979, 127.0276)),
             ("주거·고령형", "#2f8f5b", "60대 이상 비중 최고",            ("개포동", 37.4890, 127.0661)),
             ("주말·여가형", "#c8102e", "주말 인구가 평일의 약 3배",       ("양재천", 37.4840, 127.0560))],
    "춘천": [("시내 생활형", "#1f5fa8", "평일·주말 고른 생활 상권",          ("남춘천역", 37.8640, 127.7238)),
             ("주말·여가형", "#c8102e", "주말 인구가 평일의 약 2.7배",       ("명동", 37.8803, 127.7279)),
             ("낮·고령형",   "#6b3fa0", "골프장·전통시장·외곽 마을, 낮·60대+", ("중앙시장", 37.8810, 127.7255))],
}
tf = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)
gt = pd.read_csv(GT_PATH)
gj = json.load(open(ADM_PATH, encoding="utf-8"))

def dongs(sgg):
    out = []
    for f in gj["features"]:
        if sgg in f["properties"]["adm_nm"]:
            g = f["geometry"]
            for p in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
                r = np.array(p[0]); x, y = tf.transform(r[:, 0], r[:, 1]); out.append(np.c_[x, y])
    return out
SHAPE = {"강남": dongs("강남구"), "춘천": dongs("춘천시")}

def extent(region, d):
    if region == "강남":
        allp = np.vstack(SHAPE[region]); return (allp[:, 0].min(), allp[:, 0].max()), (allp[:, 1].min(), allp[:, 1].max())
    x0, x1 = d["X_COORD"].quantile([.01, .99]); y0, y1 = d["Y_COORD"].quantile([.01, .99])   # 춘천은 상권 있는 곳 위주
    return (x0 - 1500, x1 + 1500), (y0 - 1500, y1 + 1500)

fig, axes = plt.subplots(2, 4, figsize=(22, 13))
for row, region in enumerate(["강남", "춘천"]):
    d = gt[gt["region"] == region]
    (x0, x1), (y0, y1) = extent(region, d)
    for col in range(4):
        ax = axes[row, col]
        ax.set_xticks([]); ax.set_yticks([]); [s.set_visible(False) for s in ax.spines.values()]
        if col >= len(TYPES[region]):
            ax.axis("off"); continue
        t, color, desc, (lname, lat, lon) = TYPES[region][col]
        ax.add_collection(PatchCollection([MPoly(r, closed=True) for r in SHAPE[region]], facecolor=LAND, edgecolor="white", lw=0.8, zorder=1))
        rest, me = d[d["type"] != t], d[d["type"] == t]
        sz = 7 if region == "강남" else 9
        ax.scatter(rest["X_COORD"], rest["Y_COORD"], s=sz, marker="s", color=DOT, lw=0, zorder=2)
        ax.scatter(me["X_COORD"], me["Y_COORD"], s=sz * 1.6, marker="s", color=color, lw=0, zorder=3)
        if t == "낮·고령형":
            g = d[d["has_golf"] == 1]
            ax.scatter(g["X_COORD"], g["Y_COORD"], s=90, marker="^", color=color, edgecolor="white", lw=1, zorder=4)
        lx, ly = tf.transform(lon, lat)
        ax.annotate(lname, (lx, ly), xytext=(14, 14), textcoords="offset points", fontsize=11, fontweight="bold", color=color, zorder=6,
                    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=color, lw=1), arrowprops=dict(arrowstyle="-", color=color, lw=1))
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect("equal")
        share = len(me) / len(d)
        ax.set_title(f"{t}  ({len(me):,}칸, {share:.0%})", fontsize=15, fontweight="bold", color=color, loc="left", pad=22)
        ax.text(0, 1.015, desc, transform=ax.transAxes, fontsize=11.5, color=SUB, va="bottom")
    axes[row, 0].text(-0.08, 0.5, region, transform=axes[row, 0].transAxes, rotation=90, fontsize=22, fontweight="bold", color=INK, va="center", ha="center")

fig.text(0.02, 0.975, "같은 상권이라도 성격이 다르다: 유동인구 패턴으로 나눈 상권 유형", fontsize=22, fontweight="bold", color=INK)
fig.text(0.02, 0.948, "한 칸 = 상가가 있는 50m 격자.  유형마다 해당 칸만 색으로, 나머지는 회색.  유형은 '언제·누가 다니는가'(SK 유동인구)로 K-평균 군집.", fontsize=12.5, color=SUB)
plt.tight_layout(rect=[0.02, 0, 1, 0.93])
fig.savefig(SAVE, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()


## 12. 유형별 특징표

In [ ]:
ORDER = {r: [t[0] for t in TYPES[r]] for r in TYPES}

SHOW = {"pop": "유동인구\n(명/일)", "weekend_ratio": "주말÷평일", "work_share": "낮 시간\n(9~17시)",
        "night_share": "저녁\n(18~23시)", "elder_share": "60대 이상", "food_share": "음식·카페\n비중"}

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [4, 3]})
for ax, region in zip(axes, ["강남", "춘천"]):
    d = out[out["region"] == region]
    prof = d.groupby("type")[list(SHOW)].median().loc[ORDER[region]]
    z = (prof - prof.mean()) / prof.std()          # 유형끼리 비교한 상대 크기 (색)
    ax.imshow(z.values, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
    ax.set_xticks(range(len(SHOW))); ax.set_xticklabels(SHOW.values(), fontsize=10)
    ax.set_yticks(range(len(prof))); ax.set_yticklabels([f"{t} ({(d['type'] == t).sum():,})" for t in prof.index], fontsize=10)
    for i in range(prof.shape[0]):
        for j, col in enumerate(SHOW):
            v = prof.iloc[i, j]
            txt = f"{v:,.0f}" if col == "pop" else (f"{v:.2f}" if col == "weekend_ratio" else f"{v:.0%}")
            ax.text(j, i, txt, ha="center", va="center", fontsize=10,
                    color="white" if abs(z.iloc[i, j]) > 1 else "black")
    ax.set_title(f"{region} 상권 유형별 특징 (중앙값, 빨강 = 다른 유형보다 높음)", fontsize=12)
plt.tight_layout()
plt.savefig(FIG / "14_grid_type_profile.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. 확인: 명동 옆 낮·고령형 덩어리의 정체
결과: 의류·침구 상점가, 중앙로·중앙로77번길 → 춘천 중앙시장(낭만시장) 일대

In [ ]:
mx, my = tf.transform(127.7279, 37.8803)     # 명동
near = out[(out["region"] == "춘천") & (out["type"] == "낮·고령형") &
           (np.hypot(out["X_COORD"] - mx, out["Y_COORD"] - my) < 800)]   # 명동 반경 800m
print("명동 반경 800m 안의 낮·고령형 격자:", len(near), "칸")

keys = near[["X_COORD", "Y_COORD"]]
shops = sg.merge(keys, on=["X_COORD", "Y_COORD"])
shops = shops[shops["출처"] == "상가정보"]
print("\n가게 수:", len(shops))
print("\n업종 (상위 10):")
print(shops["상권업종소분류명"].value_counts().head(10))
print("\n주소 (도로명 상위 5):")
print(shops["도로명주소"].str.extract(r"(\S+로\S*)")[0].value_counts().head(5))